# 06. Mutating while looping

**Tier:** Intermediate. **Why you'd use this:** You have a list of orders and want to drop the cancelled ones. A `for` loop that calls `remove` on the list it is walking skips items, and Python raises no error. You get a wrong list that looks plausible.

**After this lesson you can:** filter a list without skipping items; spot a built-in name you hid by accident; loop over items directly instead of by index.
About 25 minutes. **Uses:** [Lists](../04-collections/01-lists-lesson.ipynb), [Loops](../03-control-flow/02-loops-lesson.ipynb), [Comprehensions](../07-iteration/01-comprehensions-lesson.ipynb)

## Quick review

1. Why does `[[0] * 3] * 2` give two rows that change together?
2. What does `is` compare?

<details><summary>Answers</summary>

The `*` repeats one reference to the same inner list. Identity: whether two names point at the same object.
</details>

## The idea

A `for` loop over a list walks by position. It keeps a counter, hands you item 0, then item 1, and so on until the counter passes the end. If you delete an item while the loop is running, every later item slides one place left. The counter still moves forward by one, so the item that slid into the gap is never visited.

The fix is to stop editing the list you are walking. Build a new list that holds only what you keep, or walk over a copy and edit the original.

Two habits come from the same root: working with positions when you do not need them. `range(len(items))` makes you fetch each item by index, while looping over `items` directly is shorter and cannot run off the end. And a name like `list` or `max` is only a name: assigning to it hides the built-in until you delete your name.

<details><summary>Explain more</summary>

`for item in items[:]` walks a copy, so removing from `items` is safe. A comprehension is usually clearer because it never edits anything. Use `enumerate(items)` when you do need the position.
</details>

## See it work

Drop the cancelled orders and leave the original list alone.

In [1]:
def keep_open(orders):
    # 1. Build a new list of the orders you want to keep.
    return [order for order in orders if order != "cancelled"]


orders = ["ok", "cancelled", "cancelled", "ok"]
print(keep_open(orders))
print(orders)

['ok', 'ok']
['ok', 'cancelled', 'cancelled', 'ok']


## Check yourself

Answer these before you run the cell.

1. What does `items` hold after the loop?
2. Which `"x"` was skipped, and why?

In [2]:
items = ["a", "x", "x", "b"]
for item in items:
    if item == "x":
        items.remove(item)
print(items)

['a', 'x', 'b']


<details><summary>Check your answers</summary>

`['a', 'x', 'b']`. The second `"x"` slid into the slot the loop had just left, so the counter moved past it.
</details>

## Why it works this way

A list iterator stores two things: the list and the next index to read. Each pass it returns the item at that index and adds one. It never checks whether the list changed. `remove` shifts every later item left by one place, so the index and the contents drift apart and one item goes unvisited. Deep dive: [The iteration protocol](../../internals/07-iteration/01-protocol-lesson.ipynb).

Dicts and sets behave differently. They keep a size counter, and their iterators compare it on every step. If the size changes mid-loop, Python raises `RuntimeError`. Lists have no such guard, which is why the list bug is silent.

The cost is not only correctness. `remove` searches from the front and then shifts the tail, so each call is O(n), and a loop that removes many items is O(n²). A comprehension touches each item once and is O(n).

Names follow the scope rules. When Python meets `sum`, it looks in the function, then the module, then the built-ins. Assigning `sum = 0` puts a name in a nearer scope, so a later `sum(prices)` finds your integer first and fails with `TypeError`. `del sum` removes your name and the built-in shows through again. Related lesson: [Scope](../05-functions/03-scope-lesson.ipynb).

Indexing with `range(len(items))` builds a counter and then does a lookup per item. It invites off-by-one slips, and it has the same shifting bug if the list changes underneath it. The design reason for `for item in items` is that the loop owns the position and you own the item.

In [3]:
import builtins

# A dict notices the change; a list does not.
scores = {"a": 1, "b": 2}
try:
    for key in scores:
        scores[key + "!"] = 0
except RuntimeError as err:
    print(err)

sum = 10
print(sum is builtins.sum)
del sum
print(sum is builtins.sum)

dictionary changed size during iteration
False
True


## When to use it

Build a new list with a comprehension when you are filtering. Walk a copy (`items[:]` or `list(items)`) when other code holds the same list and must see the change. Loop over a copy of the keys when you need to add or remove dict entries. Loop over the items directly, and use `enumerate` when you need the position. Don't remove items from the list you are walking. Don't name a variable after a built-in; pick `total`, `values`, or `biggest` instead.

## Common mistake

Removing items from a list inside the loop that walks it, then trusting the result because no error appeared.

In [4]:
lines = ["a", "", "", "b"]
for line in lines:
    if line == "":
        lines.remove(line)
print(lines)

['a', '', 'b']


One blank line is still there. The second blank slid into the slot the loop had just left. Build a new list instead.

In [5]:
lines = ["a", "", "", "b"]
lines = [line for line in lines if line != ""]
print(lines)

['a', 'b']


## Try it

Change `keep_open` so it edits the original list in place and returns nothing. Hint: assign to the whole slice, `orders[:] = ...`. Then print `orders` to confirm the caller sees the change.

## Recap

- A list loop walks by index, and `remove` shifts the items under it.
- Build a new list, or walk a copy, instead of editing the list you walk.
- A built-in name you assign to stays hidden until you `del` it.
- Loop over items directly; use `enumerate` for positions.

**Next:** [practice](06-mutating-while-looping-practice.ipynb)